# Week 2 · Shapes and slicing, properly

Your quiz log said A5 (`a[:, 0]`) and A6 (how shapes change after an operation) needed a look back. Shapes are where most real ML bugs come from, so this notebook gives you **three rules** that predict every shape, then drills until predicting shapes is automatic. It uses the matrix from your own comment in [Week 1 · NumPy §5 ↗](../week1/02_numpy_lesson.ipynb), `np.arange(1, 10).reshape(3, 3)`.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **axis**: One direction of an array. Axis 0 goes down the rows; axis 1 goes across the columns. `ndim` = the number of axes.
- **shape**: One number per axis: how long the array is in that direction. `(3, 4)` means axis 0 has length 3, and axis 1 has length 4.
- **1-D array**: An array with one axis, shape `(n,)`. It's neither a row nor a column: it has no orientation.

## 1. Reading `a[rows, cols]` one piece at a time

In [2]:
import numpy as np
a = np.arange(1, 10).reshape(3, 3)
print(a)
# → [[1 2 3]
#    [4 5 6]
#    [7 8 9]]

[[1 2 3]
 [4 5 6]
 [7 8 9]]


Inside the brackets there is **one slot per axis**, separated by a comma: `a[ which rows , which columns ]`. In each slot, `:` means "all of them", a number means "just this one", and `i:j` means "from `i` up to, not including, `j`".

**`a[:, 0]` read slowly:** rows = `:` = all rows (0, 1, 2); columns = `0` = only column 0. So pick column 0 from every row:

You see this, with the part `a[:, 0]` selects marked:

```text
              col 0   col 1   col 2
row 0  [    ( 1 )     2       3   ]
row 1  [    ( 4 )     5       6   ]
row 2  [    ( 7 )     8       9   ]
             ↑
     rows ":" = all of them, col 0 = only this one
```

- First slot `:`: go through every row, 0, 1, 2.
- Second slot `0`: in each row, take the value in column 0.
- Collect them in order: 1, then 4, then 7, so the result is `[1 4 7]`.

| row | column | value |
|---|---|---|
| 0 | 0 | `a[0, 0]` = 1 |
| 1 | 0 | `a[1, 0]` = 4 |
| 2 | 0 | `a[2, 0]` = 7 |

In [ ]:
print(a[:, 0])     # → [1 4 7]   one value from each row, all in column 0
print(a[0, :])     # → [1 2 3]   one value from each column, all in row 0 (same as a[0])
print(a[1:, :2])   # rows 1–2, columns 0–1
# → [[4 5]
#    [7 8]]

**Quick check:** What is `a[2, 1:]`?

<details><summary>Answer</summary>

`[8 9]`: row 2, columns 1 to the end.

</details>

## 2. The three shape rules

**Rule 1, indexing: a number removes that axis; a slice keeps it.** Even a slice of length 1 keeps its axis. That's the whole difference between `a[:, 0]` and `a[:, 0:1]`:

Both of these pick the same three numbers. What differs is the **shape** of the result:

```text
a[:, 0]    → column 0 as a NUMBER          a[:, 0:1]  → columns 0 up to 1 as a SLICE
             axis 1 is gone                             axis 1 is kept (length 1)

             [1 4 7]                                    [[1]
                                                         [4]
             shape (3,)                                  [7]]
             a flat line, no rows or columns
                                                        shape (3, 1)
                                                        3 rows × 1 column
```

- `0` says "exactly column 0": there's no column direction left to keep, so NumPy drops that axis.
- `0:1` says "the columns from 0 up to 1": it's still a *range* of columns, which happens to contain one column, so the column axis stays.
- Count the axes that are left: `a[:, 0]` keeps only the row axis, giving `(3,)`; `a[:, 0:1]` keeps both, giving `(3, 1)`.

| Expression | Slot by slot | Result | Shape |
|---|---|---|---|
| `a` |  | the whole matrix | `(3, 3)` |
| `a[0]` | row: number → axis 0 removed | `[1 2 3]` | `(3,)` |
| `a[:, 0]` | rows kept, column: number → axis 1 removed | `[1 4 7]` | `(3,)` |
| `a[:, 0:1]` | rows kept, column: slice → axis 1 **kept** | `[[1] [4] [7]]` | `(3, 1)` |
| `a[:, [0]]` | a list of indices also keeps the axis | `[[1] [4] [7]]` | `(3, 1)` |
| `a[0, 0]` | both numbers → both axes removed | `1` | `()` (a single number) |
| `a[a > 4]` | a mask picks elements → always 1-D | `[5 6 7 8 9]` | `(5,)` |

In [ ]:
for expr in ["a[0]", "a[:, 0]", "a[:, 0:1]", "a[:, [0]]", "a[0, 0]", "a[a > 4]"]:
    print(f"{expr:10s} shape {np.shape(eval(expr))}")
# → a[0]       shape (3,)
#   a[:, 0]    shape (3,)
#   a[:, 0:1]  shape (3, 1)
#   a[:, [0]]  shape (3, 1)
#   a[0, 0]    shape ()
#   a[a > 4]   shape (5,)

**Rule 2, reductions: the axis you reduce disappears** (unless you pass `keepdims=True`, which keeps it with length 1). From shape `(3, 4)`:

Take `a = [[1, 2, 3], [4, 5, 6]]`, shape `(2, 3)`:

```text
sum(axis=0): squash DOWN the rows           sum(axis=1): squash ACROSS the columns

   [ 1   2   3 ]                               [ 1   2   3 ]  →  1+2+3 =  6
   [ 4   5   6 ]                               [ 4   5   6 ]  →  4+5+6 = 15
     ↓   ↓   ↓
   [ 5   7   9 ]   one total per column        [ 6  15 ]     one total per row

   (2, 3) → (3,)   axis 0 (the 2) is gone      (2, 3) → (2,)   axis 1 (the 3) is gone
```

- `axis=0` walks along axis 0, i.e. down a column: $1 + 4 = 5$, $2 + 5 = 7$, $3 + 6 = 9$.
- `axis=1` walks along axis 1, i.e. across a row: $1 + 2 + 3 = 6$, $4 + 5 + 6 = 15$.
- The axis you walked along is used up, so it disappears from the shape.

| Operation | Result shape | In words |
|---|---|---|
| `x.sum()` | `()` | everything → one number |
| `x.sum(axis=0)` | `(4,)` | per column (axis 0 disappears) |
| `x.sum(axis=1)` | `(3,)` | per row (axis 1 disappears) |
| `x.mean(axis=1, keepdims=True)` | `(3, 1)` | per row, but kept as a column so it can broadcast back |
| `x.argmax(axis=1)` | `(3,)` | the position of the max in each row | <----------- you made a mistake here. Dont make mistakes brother :(

**Rule 3, combining two arrays:**

| Operation | Shapes | Result | Rule |
|---|---|---|---|
| element-wise `+ - * /` | `(3, 4)` and `(3, 4)` | `(3, 4)` | same shape → same shape |
| broadcasting | `(3, 4)` and `(4,)` | `(3, 4)` | line up from the right; sizes must be equal or 1 |
| broadcasting | `(3, 4)` and `(3, 1)` | `(3, 4)` | the 1 stretches to 4 |
| matrix product `@` | `(3, 4)` and `(4, 2)` | `(3, 2)` | inner sizes match and disappear |
| matrix × vector | `(3, 4)` and `(4,)` | `(3,)` | one dot product per row |

And the operations that only rearrange: `reshape` keeps the total number of elements, and `.T` reverses the order of the axes: `(3, 4)` becomes `(4, 3)`.

In [ ]:
x = np.ones((3, 4))
print(x.sum(axis=0).shape, x.mean(axis=1, keepdims=True).shape, (x @ np.ones((4, 2))).shape, (x @ np.ones(4)).shape)
# → (4,) (3, 1) (3, 2) (3,)

**What broadcasting actually does**

`X + row` with `X = [[1, 2, 3], [4, 5, 6]]` (shape `(2, 3)`) and `row = [10, 20, 30]` (shape `(3,)`):

```text
Step 1: line the shapes up from the right

      X:  (2, 3)
    row:     (3,)      3 vs 3 → equal ✓      2 vs (nothing) → fine ✓

Step 2: NumPy stretches row to the missing row axis (without copying memory)

    [ 10  20  30 ]       becomes      [ 10  20  30 ]
                                      [ 10  20  30 ]

Step 3: add element by element

    [ 1   2   3 ]     [ 10  20  30 ]     [ 11  22  33 ]
    [ 4   5   6 ]  +  [ 10  20  30 ]  =  [ 14  25  36 ]
```

Entry `[1, 2]` of the result: row 1 of `X` has `6` in column 2, the stretched row has `30` in column 2, so $6 + 30 = 36$.

## 3. `(4,)` vs `(1, 4)` vs `(4, 1)`

In [Week 1 · NumPy §10 ↗](../week1/02_numpy_lesson.ipynb) you replaced a cell with your own test, `np.ones(4)`, and got shape `(4,)`. That's a **1-D array**: it has no orientation, so it's neither a row nor a column. `(1, 4)` is a **row** (one row, four columns) and `(4, 1)` is a **column** (four rows, one column).

In [ ]:
v = np.array([1, 2, 3, 4])
print(v.shape, v.T.shape)              # → (4,) (4,)        .T does nothing to a 1-D array!
row = v.reshape(1, -1)                 # → shape (1, 4)
col = v.reshape(-1, 1)                 # → shape (4, 1)     same as v[:, None]
print(row.shape, col.shape, v[:, None].shape)   # → (1, 4) (4, 1) (4, 1)
print((col + row).shape)               # → (4, 4)   a column + a row broadcasts to a full grid

> ⚠️ **Common mistake.** **The most common regression bug.** If `y` has shape `(n,)` but your predictions have shape `(n, 1)`, then `y - pred` broadcasts to shape `(n, n)`. The code runs, there's no error, and the loss is silently wrong. Always check that both have the same shape.

Here's why `y - pred` turns into a $3 \times 3$ grid when `y` is `(3,)` and `pred` is `(3, 1)`:

```text
Line up the shapes from the right:   y: (3,)    pred: (3, 1)
   last axis:  3 vs 1 → pred's column stretches to 3 columns
   next axis:  (nothing) vs 3 → y is repeated down 3 rows

        y repeated down           pred stretched across        y - pred
     [ 1   2   3 ]               [ 1   1   1 ]              [  0   1   2 ]
     [ 1   2   3 ]       -       [ 2   2   2 ]      =       [ -1   0   1 ]
     [ 1   2   3 ]               [ 3   3   3 ]              [ -2  -1   0 ]
```

- You wanted 3 differences, one per sample: $1 - 1$, $2 - 2$, $3 - 3$, all 0. Those are only the diagonal.
- Instead you got all 9 pairs. Their squares are $0, 1, 4, 1, 0, 1, 4, 1, 0$, which sum to 12.
- So the "MSE" is $\frac{12}{9} = 1.333$ instead of 0, and nothing warned you.

In [ ]:
y = np.array([1.0, 2.0, 3.0])            # (3,)
pred = np.array([[1.0], [2.0], [3.0]])   # (3, 1): looks the same, isn't
print((y - pred).shape)                  # → (3, 3)   !!
print(np.mean((y - pred) ** 2))          # → 1.3333333333333333   should be 0
print(np.mean((y - pred.ravel()) ** 2))  # → 0.0      .ravel() flattens (3, 1) to (3,)

## 4. Joining arrays

You'll need this for regression: adding a column of 1s for the intercept. `np.column_stack` puts 1-D arrays side by side as columns. `np.concatenate` joins along an existing axis.

```text
np.ones(3) = [1 1 1]       x = [1 2 3]

column_stack stands each 1-D array up as a column, then puts them side by side:

    [ 1 ]   [ 1 ]          [ 1   1 ]
    [ 1 ]   [ 2 ]    →     [ 1   2 ]
    [ 1 ]   [ 3 ]          [ 1   3 ]

    (3,)    (3,)            (3, 2)
```

In [ ]:
x = np.array([1.0, 2.0, 3.0])
X = np.column_stack([np.ones(3), x])     # a column of 1s, then the feature
print(X)
# → [[1. 1.]
#    [1. 2.]
#    [1. 3.]]
A = np.ones((2, 3)); B = np.zeros((2, 3))
print(np.concatenate([A, B], axis=0).shape, np.concatenate([A, B], axis=1).shape)   # → (4, 3) (2, 6)

## 5. Shape drills

Say each answer out loud before opening it. `X` has shape `(100, 5)`, `w` has shape `(5,)`, and `y` has shape `(100,)`.

**Quick check:** `X[0].shape`

<details><summary>Answer</summary>

`(5,)`: one sample, all 5 features.

</details>

**Quick check:** `X[:, 0].shape`

<details><summary>Answer</summary>

`(100,)`: one feature for every sample.

</details>

**Quick check:** `X[:, :2].shape`

<details><summary>Answer</summary>

`(100, 2)`

</details>

**Quick check:** `(X @ w).shape`

<details><summary>Answer</summary>

`(100,)`: one prediction per sample.

</details>

**Quick check:** `(X.T @ X).shape`

<details><summary>Answer</summary>

`(5, 5)`

</details>

**Quick check:** `(X.T @ (X @ w - y)).shape`

<details><summary>Answer</summary>

`(5,)`: the gradient has one entry per weight, the same shape as `w`.

</details>

**Quick check:** `X.mean(axis=0).shape`

<details><summary>Answer</summary>

`(5,)`: the mean of each feature.

</details>

**Quick check:** `(X - X.mean(axis=0)).shape`

<details><summary>Answer</summary>

`(100, 5)`: broadcasting subtracts each column's mean.

</details>

**Quick check:** `X[y > 0].shape`

<details><summary>Answer</summary>

`(k, 5)`, where `k` is how many entries of `y` are positive: a mask on axis 0 keeps whole rows.

</details>

**Quick check:** `np.column_stack([np.ones(100), X]).shape`

<details><summary>Answer</summary>

`(100, 6)`: the intercept column plus 5 features.

</details>

**Your turn.** Check all ten answers with code. Make `X`, `w`, `y` with `np.ones` or a random generator, then print each shape.

<details><summary>Show one answer</summary>

e.g. `print(X[0].shape, X[:, 0].shape, (X @ w).shape, ...)`

</details>

In [ ]:
X = np.ones((100, 5)); w = np.ones(5); y = np.linspace(-1, 1, 100)
# your code here

---
Next: `04_ml_intro_linear_regression.ipynb`, where machine learning starts.